In [2]:
# Single-cell sanity check:
# 2NN / Participation Ratio dimensionality estimates for CIFAR-10 cats
# under tiny L_inf pixel noise perturbations.
!pip install -q git+https://github.com/fra31/auto-attack

import numpy as np
import matplotlib.pyplot as plt

from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

import torch
import torchvision
import torchvision.transforms as T
import torch.nn.functional as F


# ----------------------------
# Config
# ----------------------------

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

N_BASE = 500          # number of clean cat images
K_PER_IMAGE = 20      # noisy variants per base image
EPS_LIST = [0, 1/255, 2/255, 4/255, 8/255]
MAX_POINTS_2NN = 10000

DO_PCA_PREPROCESS = False
PCA_DIM = 256         # optional; leave off for raw-pixel estimates

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)


# ----------------------------
# Load CIFAR-10 cats
# ----------------------------

transform = T.ToTensor()
dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

# CIFAR-10 labels:
# 0 airplane, 1 automobile, 2 bird, 3 cat, 4 deer,
# 5 dog, 6 frog, 7 horse, 8 ship, 9 truck
CAT_LABEL = 3

cat_imgs = []
for img, label in dataset:
    if label == CAT_LABEL:
        cat_imgs.append(img)
    if len(cat_imgs) >= N_BASE:
        break

X_base_torch = torch.stack(cat_imgs).to(DEVICE)  # [N, 3, 32, 32]
X_base = X_base_torch.detach().cpu().numpy()
N, C, H, W = X_base.shape
D = C * H * W

print(f"Loaded {N} cat images with ambient pixel dimension D = {D}.")


# ----------------------------
# Dimensionality estimators
# ----------------------------

def participation_ratio(X):
    """
    PR = (sum eigenvalues)^2 / sum eigenvalues^2
    where eigenvalues are covariance eigenvalues.
    """
    X = np.asarray(X, dtype=np.float64)
    Xc = X - X.mean(axis=0, keepdims=True)

    # Use SVD instead of explicitly forming giant covariance.
    # Cov eigenvalues are proportional to singular_values^2.
    s = np.linalg.svd(Xc, full_matrices=False, compute_uv=False)
    eig = s**2 / max(len(X) - 1, 1)

    denom = np.sum(eig**2)
    if denom == 0:
        return 0.0
    return float((np.sum(eig)**2) / denom)


def twonn_dimension(X, max_points=MAX_POINTS_2NN, seed=SEED):
    """
    2NN estimator from Facco et al.
    Uses ratio mu = r2 / r1.
    CDF relation: F(mu) = 1 - mu^{-d}
    So log(1-F(mu)) = -d log(mu).
    Estimate slope through origin on sorted empirical CDF.

    This is intentionally simple and transparent.
    """
    X = np.asarray(X, dtype=np.float64)

    # Subsample if needed.
    if len(X) > max_points:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(X), size=max_points, replace=False)
        X = X[idx]

    # Remove exact duplicates or near duplicates can break r1=0.
    # Add ultra-tiny jitter to avoid division by zero in duplicated control.
    X = X + 1e-12 * np.random.default_rng(seed).standard_normal(X.shape)

    nbrs = NearestNeighbors(n_neighbors=3, algorithm="auto", metric="euclidean")
    nbrs.fit(X)
    distances, _ = nbrs.kneighbors(X)

    # distances[:, 0] is self-distance.
    r1 = distances[:, 1]
    r2 = distances[:, 2]

    valid = (r1 > 0) & (r2 > r1)
    mu = r2[valid] / r1[valid]

    if len(mu) < 10:
        return np.nan

    mu = np.sort(mu)
    n = len(mu)

    # Empirical CDF values avoiding endpoint 1.
    F_emp = (np.arange(1, n + 1) - 0.5) / n

    x = np.log(mu)
    y = -np.log(1 - F_emp)

    # Robust-ish trimming: extremes can be noisy.
    lo = int(0.05 * n)
    hi = int(0.95 * n)
    x_fit = x[lo:hi]
    y_fit = y[lo:hi]

    # Fit y = d*x through origin.
    d_hat = np.sum(x_fit * y_fit) / np.sum(x_fit * x_fit)

    return float(d_hat)


def maybe_pca(X, dim=PCA_DIM):
    if not DO_PCA_PREPROCESS:
        return X

    dim = min(dim, X.shape[0] - 1, X.shape[1])
    pca = PCA(n_components=dim, random_state=SEED)
    return pca.fit_transform(X)


def estimate_dims(name, X):
    """
    X should be [num_samples, num_features].
    """
    X_proc = maybe_pca(X)
    d_2nn = twonn_dimension(X_proc)
    d_pr = participation_ratio(X_proc)
    print(f"{name:35s} | n={len(X):5d} | 2NN={d_2nn:9.2f} | PR={d_pr:9.2f}")
    return {
        "condition": name,
        "n": len(X),
        "twonn": d_2nn,
        "pr": d_pr,
    }


# ----------------------------
# Data generation helpers
# ----------------------------

def flatten_torch(x):
    return x.detach().cpu().numpy().reshape(len(x), -1)


def make_noisy_variants(X_base_torch, eps, k_per_image=K_PER_IMAGE):
    """
    For each base image x, generate k samples:
        clamp(x + Uniform[-eps, eps], 0, 1)
    """
    N = X_base_torch.shape[0]
    x = X_base_torch[:, None, :, :, :].repeat(1, k_per_image, 1, 1, 1)
    x = x.reshape(N * k_per_image, C, H, W)

    if eps > 0:
        noise = torch.empty_like(x).uniform_(-eps, eps)
        x = torch.clamp(x + noise, 0.0, 1.0)

    return x


def lowpass_batch(x, kernel_size=5):
    """
    Simple average-pooling lowpass, preserving image size.
    """
    pad = kernel_size // 2
    return F.avg_pool2d(
        F.pad(x, (pad, pad, pad, pad), mode="reflect"),
        kernel_size=kernel_size,
        stride=1,
    )


# ----------------------------
# Main sanity checks
# ----------------------------

results = []

print("\nDimensionality estimates")
print("-" * 75)

# 1. Clean cats
X_clean = X_base.reshape(N, -1)
results.append(estimate_dims("clean cats", X_clean))

# 2. Repeated clean cats: same images repeated K times.
# This should not truly add dimension, but exact duplicates make nearest-neighbor estimators weird.
X_repeated = np.repeat(X_clean, K_PER_IMAGE, axis=0)
results.append(estimate_dims("repeated clean cats", X_repeated))

# 3. Noisy cats for multiple epsilons
for eps in EPS_LIST:
    X_noisy_t = make_noisy_variants(X_base_torch, eps)
    X_noisy = flatten_torch(X_noisy_t)
    results.append(estimate_dims(f"cats + U[-{eps:.5f},{eps:.5f}]", X_noisy))

# 4. Low-pass-smoothed noisy cats
for eps in EPS_LIST:
    X_noisy_t = make_noisy_variants(X_base_torch, eps)
    X_lp_t = lowpass_batch(X_noisy_t, kernel_size=5)
    X_lp = flatten_torch(X_lp_t)
    results.append(estimate_dims(f"lowpass(cats + eps={eps:.5f})", X_lp))

# 5. Pure local noise cloud around one fixed cat.
# This isolates the "tiny cube around one image" effect.
x0 = X_base_torch[:1]
for eps in EPS_LIST[1:]:
    x0_rep = x0.repeat(N_BASE * K_PER_IMAGE, 1, 1, 1)
    noise = torch.empty_like(x0_rep).uniform_(-eps, eps)
    X_cloud_t = torch.clamp(x0_rep + noise, 0.0, 1.0)
    X_cloud = flatten_torch(X_cloud_t)
    results.append(estimate_dims(f"single cat local cube eps={eps:.5f}", X_cloud))


# ----------------------------
# Plot results
# ----------------------------

conditions = [r["condition"] for r in results]
twonn_vals = [r["twonn"] for r in results]
pr_vals = [r["pr"] for r in results]

plt.figure(figsize=(12, 5))
plt.bar(np.arange(len(results)) - 0.2, twonn_vals, width=0.4, label="2NN")
plt.bar(np.arange(len(results)) + 0.2, pr_vals, width=0.4, label="Participation ratio")
plt.xticks(np.arange(len(results)), conditions, rotation=75, ha="right")
plt.ylabel("Estimated dimension")
plt.title("Dimensionality estimates for clean/noisy CIFAR-10 cat images")
plt.legend()
plt.tight_layout()
plt.show()


# ----------------------------
# Focused epsilon plot
# ----------------------------

def get_result(prefix):
    xs, ys_2nn, ys_pr = [], [], []
    for r in results:
        if r["condition"].startswith(prefix):
            # parse eps from name crudely
            cond = r["condition"]
            if "eps=" in cond:
                eps_str = cond.split("eps=")[1].split(")")[0].split()[0]
                eps = float(eps_str)
            elif "U[-" in cond:
                eps_str = cond.split("U[-")[1].split(",")[0]
                eps = float(eps_str)
            else:
                continue
            xs.append(eps * 255)
            ys_2nn.append(r["twonn"])
            ys_pr.append(r["pr"])
    order = np.argsort(xs)
    return np.array(xs)[order], np.array(ys_2nn)[order], np.array(ys_pr)[order]


x_noisy, y_noisy_2nn, y_noisy_pr = get_result("cats + U")
x_lp, y_lp_2nn, y_lp_pr = get_result("lowpass")

plt.figure(figsize=(8, 5))
plt.plot(x_noisy, y_noisy_2nn, marker="o", label="2NN: noisy cats")
plt.plot(x_noisy, y_noisy_pr, marker="o", label="PR: noisy cats")
plt.plot(x_lp, y_lp_2nn, marker="o", label="2NN: lowpass noisy cats")
plt.plot(x_lp, y_lp_pr, marker="o", label="PR: lowpass noisy cats")
plt.xlabel("epsilon in units of /255")
plt.ylabel("Estimated dimension")
plt.title("Effect of tiny L∞ pixel noise on estimated dimension")
plt.legend()
plt.tight_layout()
plt.show()


# ----------------------------
# Show example images
# ----------------------------

eps_show = 4 / 255
X_show_clean = X_base_torch[:8]
X_show_noisy = make_noisy_variants(X_base_torch[:8], eps_show, k_per_image=1)
X_show_lp = lowpass_batch(X_show_noisy, kernel_size=5)

grid = torchvision.utils.make_grid(
    torch.cat([X_show_clean, X_show_noisy, X_show_lp], dim=0),
    nrow=8,
    padding=2,
)

plt.figure(figsize=(12, 5))
plt.imshow(np.transpose(grid.detach().cpu().numpy(), (1, 2, 0)))
plt.axis("off")
plt.title("Rows: clean cats / cats + ±4/255 noise / lowpass(noisy cats)")
plt.show()


# ----------------------------
# Interpretive reminder
# ----------------------------

print("\nInterpretation notes:")
print("""
1. If 2NN jumps a lot for cats + ±4/255 noise, that means the estimator is seeing
   tiny pixel-level nuisance degrees of freedom, not merely semantic cat variation.

2. The 'single cat local cube' condition is the sharpest diagnostic:
   it has no semantic variation at all, only tiny pixel perturbations around one image.

3. Low-pass noisy cats test whether the inflation is specifically high-frequency,
   independent pixel noise.

4. This does not refute the PM definition if PM means the model's high-confidence
   input region. But it does challenge interpreting PM dimension as the dimension
   of a clean human-semantic class manifold.
""")

  Preparing metadata (setup.py) ... done
Device: cuda


 19%|█▉        | 32.3M/170M [00:03<00:15, 9.10MB/s]


KeyboardInterrupt: 

In [3]:
!pip install -q git+https://github.com/fra31/auto-attack

  Preparing metadata (setup.py) ... done


In [ ]:
# Single Colab cell:
# Uniform/smooth/noise initialization sanity check for PM dimensionality.
#
# Goal:
#   Compare PM dimensionality estimates when optimizing to "cat" from:
#     1. random RGB uniform backgrounds
#     2. fixed gray + tiny jitter
#     3. low-frequency smooth random fields
#     4. iid pixel noise
#     5. natural CIFAR cat images optimized upward
#     6. natural CIFAR noncat images optimized to cat
#
# Notes:
#   - Uses RobustBench CIFAR-10 model if available.
#   - RobustBench models usually expect inputs in [0,1].
#   - This is a sanity-check replication-style probe, not an exact reproduction
#     of the paper's code.
#
# Important fixes in this version:
#   1. Natural-image initializers now sample without replacement globally.
#      Previously, each batch independently sampled from X_cat/X_noncat,
#      so repeated images could appear across batches.
#   2. 2NN now removes exact duplicate rows before estimating dimension.
#      This makes duplicate pathologies explicit instead of silently corrupting
#      r1/r2 behavior.
#   3. Optimizer now has MIN_STEPS_BEFORE_MASKING, so images already classified
#      as cat at initialization still get some optimization. This helps avoid
#      unchanged-looking final images in the solid-color condition.

import sys, subprocess, os, math, time, warnings
warnings.filterwarnings("ignore")

# ----------------------------
# Install dependencies
# ----------------------------

try:
    import robustbench
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "robustbench"])

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

from sklearn.neighbors import NearestNeighbors

from robustbench.utils import load_model


# ----------------------------
# Config
# ----------------------------

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

TARGET_CLASS = 3          # CIFAR-10 cat
TARGET_NAME = "cat"

# Paper uses a high-confidence threshold around 0.75 for CIFAR.
CONF_THRESH = 0.95

# Reduce these if Colab is slow; increase for better estimates.
N_SAMPLES = 4096
BATCH_SIZE = 128
STEPS = 400
LR = 0.05

# Even samples already above threshold get optimized for at least this many steps.
# This prevents already-cat solid-color starts from staying visually unchanged.
MIN_STEPS_BEFORE_MASKING = 40

# Adam + optional small total variation penalty.
TV_WEIGHT = 0.0

# 2NN can be slow/noisy. This caps points used for 2NN.
MAX_POINTS_2NN = 5000

# RobustBench model. The paper's Fig. 2 mentions WideResNet-28-10;
# RobustBench's "Standard" CIFAR-10 model is the closest simple starting point.
# If this name fails in your environment, try:
#   "Wong2020Fast", "Rice2020Overfitting", "Engstrom2019Robustness"
MODEL_NAME = "Standard"
DATASET = "cifar10"
THREAT_MODEL = "Linf"

print(f"Loading RobustBench model: {MODEL_NAME}")
model = load_model(
    model_name=MODEL_NAME,
    dataset=DATASET,
    threat_model=THREAT_MODEL
).to(DEVICE).eval()

for p in model.parameters():
    p.requires_grad_(False)


# ----------------------------
# Load CIFAR-10 natural cats/noncats
# ----------------------------

transform = T.ToTensor()
cifar_train = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

cat_imgs = []
noncat_imgs = []

for img, y in cifar_train:
    if y == TARGET_CLASS and len(cat_imgs) < N_SAMPLES:
        cat_imgs.append(img)
    if y != TARGET_CLASS and len(noncat_imgs) < N_SAMPLES:
        noncat_imgs.append(img)
    if len(cat_imgs) >= N_SAMPLES and len(noncat_imgs) >= N_SAMPLES:
        break

X_cat = torch.stack(cat_imgs).to(DEVICE)
X_noncat = torch.stack(noncat_imgs).to(DEVICE)

C, H, W = 3, 32, 32
D = C * H * W

print(f"Ambient dimension D = {D}")
print(f"Loaded {len(X_cat)} natural cats and {len(X_noncat)} noncats.")


# ----------------------------
# Helpers
# ----------------------------

def flatten(x):
    return x.detach().cpu().numpy().reshape(len(x), -1)


def exact_duplicate_report(X, name="X"):
    """
    Quick diagnostic for exact duplicate flattened samples.
    """
    X = np.asarray(X)
    if len(X) == 0:
        return
    Xu = np.unique(X, axis=0)
    n_dup = len(X) - len(Xu)
    if n_dup > 0:
        print(f"Warning: {name} contains {n_dup} exact duplicate rows out of {len(X)}.")


# ----------------------------
# Dimensionality estimators
# ----------------------------

def participation_ratio(X):
    """
    PR = (sum covariance eigenvalues)^2 / sum covariance eigenvalues^2.
    Uses SVD for numerical stability.
    """
    X = np.asarray(X, dtype=np.float64)
    Xc = X - X.mean(axis=0, keepdims=True)

    if Xc.shape[0] < 3:
        return np.nan

    s = np.linalg.svd(Xc, full_matrices=False, compute_uv=False)
    eig = s**2 / max(Xc.shape[0] - 1, 1)

    denom = np.sum(eig**2)
    if denom <= 0:
        return np.nan

    return float((np.sum(eig)**2) / denom)


def twonn_dimension(X, max_points=MAX_POINTS_2NN, seed=SEED, verbose=True):
    """
    2NN estimator:
      F(mu) = 1 - mu^{-d}, where mu = r2/r1.
    Then:
      -log(1-F(mu)) = d log(mu).

    This version explicitly removes exact duplicate rows before estimating.
    Exact duplicates make nearest-neighbor radii degenerate and can badly break
    the estimator, especially when duplicated natural images are produced by
    independent per-batch sampling.
    """
    X = np.asarray(X, dtype=np.float64)

    if len(X) < 20:
        return np.nan

    # Remove exact duplicates before optional subsampling.
    n_before = len(X)
    X = np.unique(X, axis=0)
    n_after = len(X)

    if verbose and n_after < n_before:
        print(f"  2NN duplicate cleanup: removed {n_before - n_after} exact duplicate rows.")

    if len(X) < 20:
        return np.nan

    if len(X) > max_points:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(X), size=max_points, replace=False)
        X = X[idx]

    # Tiny jitter only after deduplication, to avoid zero-distance numerical ties.
    rng = np.random.default_rng(seed)
    X = X + 1e-12 * rng.standard_normal(X.shape)

    nbrs = NearestNeighbors(n_neighbors=3, algorithm="auto", metric="euclidean")
    nbrs.fit(X)
    distances, _ = nbrs.kneighbors(X)

    r1 = distances[:, 1]
    r2 = distances[:, 2]

    valid = (r1 > 0) & (r2 > r1) & np.isfinite(r1) & np.isfinite(r2)
    mu = r2[valid] / r1[valid]

    if len(mu) < 20:
        return np.nan

    mu = np.sort(mu)
    n = len(mu)

    # Empirical CDF with standard plotting-position offset.
    F_emp = (np.arange(1, n + 1) - 0.5) / n

    x = np.log(mu)
    y = -np.log(1 - F_emp)

    valid_fit = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x = x[valid_fit]
    y = y[valid_fit]

    if len(x) < 20:
        return np.nan

    n = len(x)
    lo = int(0.05 * n)
    hi = int(0.95 * n)

    x_fit = x[lo:hi]
    y_fit = y[lo:hi]

    denom = np.sum(x_fit * x_fit)
    if denom <= 0:
        return np.nan

    # Fit through origin.
    d_hat = np.sum(x_fit * y_fit) / denom
    return float(d_hat)


def estimate_dims(name, x):
    X = flatten(x) if torch.is_tensor(x) else np.asarray(x)
    exact_duplicate_report(X, name=name)

    d_pr = participation_ratio(X)
    d_2nn = twonn_dimension(X)

    print(f"{name:35s} | n={len(X):5d} | 2NN={d_2nn:9.2f} | PR={d_pr:9.2f}")

    return {
        "condition": name,
        "n": len(X),
        "twonn": d_2nn,
        "pr": d_pr,
    }


# ----------------------------
# Initializers
# ----------------------------

def init_random_rgb_constant(n):
    """
    Each image is a uniform RGB color, random across samples.
    Starting distribution has at most 3D variation.
    """
    rgb = torch.rand(n, 3, 1, 1, device=DEVICE)
    return rgb.expand(n, 3, H, W).clone()


def init_fixed_gray_jitter(n, gray=0.5, jitter=1/255):
    """
    Almost identical gray images, with tiny iid jitter to break exact symmetry.
    """
    x = torch.full((n, 3, H, W), gray, device=DEVICE)
    x = x + torch.empty_like(x).uniform_(-jitter, jitter)
    return x.clamp(0, 1)


def init_smooth_noise(n, grid=4):
    """
    Low-frequency random field: sample grid x grid noise, upsample to 32x32.
    """
    z = torch.rand(n, 3, grid, grid, device=DEVICE)
    x = F.interpolate(z, size=(H, W), mode="bicubic", align_corners=False)
    return x.clamp(0, 1)


def init_iid_noise(n):
    """
    Full iid pixel noise in [0,1].
    """
    return torch.rand(n, 3, H, W, device=DEVICE)


def init_natural_cats(n):
    """
    Sample natural cats without replacement.

    Important: this function should be called once for the whole condition,
    not once per batch. run_condition does that below.
    """
    if n > len(X_cat):
        raise ValueError(f"Asked for {n} cats, but only loaded {len(X_cat)}.")
    idx = torch.randperm(len(X_cat), device=DEVICE)[:n]
    return X_cat[idx].clone()


def init_natural_noncats(n):
    """
    Sample natural noncats without replacement.

    Important: this function should be called once for the whole condition,
    not once per batch. run_condition does that below.
    """
    if n > len(X_noncat):
        raise ValueError(f"Asked for {n} noncats, but only loaded {len(X_noncat)}.")
    idx = torch.randperm(len(X_noncat), device=DEVICE)[:n]
    return X_noncat[idx].clone()


# ----------------------------
# Optimization
# ----------------------------

def total_variation(x):
    return (
        (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean()
        + (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean()
    )


@torch.no_grad()
def confidence_stats(x):
    logits = model(x)
    probs = logits.softmax(dim=1)
    p = probs[:, TARGET_CLASS]
    pred = probs.argmax(dim=1)
    return {
        "mean_p": float(p.mean().item()),
        "median_p": float(p.median().item()),
        "frac_target": float((pred == TARGET_CLASS).float().mean().item()),
        "frac_conf": float((p >= CONF_THRESH).float().mean().item()),
    }


def optimize_to_target(
    x0,
    steps=STEPS,
    lr=LR,
    conf_thresh=CONF_THRESH,
    min_steps_before_masking=MIN_STEPS_BEFORE_MASKING,
    verbose=False,
):
    """
    Projected gradient descent on CE-to-target.

    Samples above threshold are only frozen after min_steps_before_masking.
    This avoids a visual artifact where solid-color samples that are already
    classified as cat stay exactly unchanged.
    """
    x0 = x0.detach().clone().to(DEVICE)
    x = x0.clone().detach().requires_grad_(True)

    y = torch.full((len(x),), TARGET_CLASS, device=DEVICE, dtype=torch.long)
    opt = torch.optim.Adam([x], lr=lr)

    for step in range(steps):
        opt.zero_grad(set_to_none=True)

        logits = model(x)
        probs = logits.softmax(dim=1)
        p_t = probs[:, TARGET_CLASS]

        ce = F.cross_entropy(logits, y, reduction="none")

        if step < min_steps_before_masking:
            active = torch.ones_like(p_t)
        else:
            active = (p_t < conf_thresh).float()

        if active.sum().item() == 0:
            break

        loss = (ce * active).sum() / active.sum().clamp_min(1.0)

        if TV_WEIGHT > 0:
            loss = loss + TV_WEIGHT * total_variation(x)

        loss.backward()
        opt.step()

        with torch.no_grad():
            x.clamp_(0, 1)

        if verbose and step % 50 == 0:
            print(
                f"step={step:4d} "
                f"mean p(cat)={p_t.mean().item():.3f} "
                f"frac >= {conf_thresh}={(p_t >= conf_thresh).float().mean().item():.3f} "
                f"active={active.mean().item():.3f}"
            )

    with torch.no_grad():
        logits = model(x)
        probs = logits.softmax(dim=1)
        p_t = probs[:, TARGET_CLASS]
        reached = p_t >= conf_thresh

    return x.detach(), reached.detach(), p_t.detach(), x0.detach()


def run_condition(name, init_fn, n=N_SAMPLES):
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    t0 = time.time()

    # CRITICAL FIX:
    # Build all initial samples once, then batch through the fixed tensor.
    # This prevents natural_cats/natural_noncats from resampling independently
    # in each batch and accidentally duplicating images across the condition.
    x0_all = init_fn(n).detach().clone().to(DEVICE)

    all_x0 = []
    all_xT = []
    all_reached = []
    all_p = []

    for start in range(0, n, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n)
        x0 = x0_all[start:end]
        xT, reached, p, x0_saved = optimize_to_target(x0)

        all_x0.append(x0_saved.cpu())
        all_xT.append(xT.cpu())
        all_reached.append(reached.cpu())
        all_p.append(p.cpu())

    x0 = torch.cat(all_x0, dim=0).to(DEVICE)
    xT = torch.cat(all_xT, dim=0).to(DEVICE)
    reached = torch.cat(all_reached, dim=0).bool()
    p = torch.cat(all_p, dim=0)

    print(f"Time: {time.time() - t0:.1f}s")
    print(f"Reached threshold: {reached.float().mean().item():.3f}")
    print(f"Mean p({TARGET_NAME}): {p.mean().item():.3f}; median: {p.median().item():.3f}")

    # Movement diagnostics on all samples.
    X0_all = flatten(x0)
    XT_all = flatten(xT)
    l2_move_all = np.linalg.norm(XT_all - X0_all, axis=1)
    print(f"Unchanged-ish samples, ||xT-x0||_2 < 1e-6: {(l2_move_all < 1e-6).mean():.3f}")
    print(f"Median movement over all samples: {np.median(l2_move_all):.6f}")

    # Use only successful samples for PM dimensionality.
    if reached.sum().item() >= 20:
        xT_succ = xT[reached.to(DEVICE)]
        x0_succ = x0[reached.to(DEVICE)]
    else:
        print("Warning: few/no samples reached threshold; using all final samples.")
        xT_succ = xT
        x0_succ = x0

    # Residual and frequency decomposition.
    residual = xT_succ - x0_succ

    xT_low = F.avg_pool2d(
        F.pad(xT_succ, (2, 2, 2, 2), mode="reflect"),
        kernel_size=5,
        stride=1,
    )
    xT_high = (xT_succ - xT_low + 0.5).clamp(0, 1)

    res = {}
    res["x0"] = estimate_dims(name + " | init x0", x0_succ)
    res["xT"] = estimate_dims(name + " | final xT", xT_succ)
    res["resid"] = estimate_dims(name + " | residual xT-x0", residual)
    res["low"] = estimate_dims(name + " | lowpass xT", xT_low)
    res["high"] = estimate_dims(name + " | highpass-ish xT", xT_high)

    # Correlation between x0 and xT per sample.
    X0 = flatten(x0_succ)
    XT = flatten(xT_succ)

    X0c = X0 - X0.mean(axis=1, keepdims=True)
    XTc = XT - XT.mean(axis=1, keepdims=True)

    corr = np.sum(X0c * XTc, axis=1) / (
        np.linalg.norm(X0c, axis=1) * np.linalg.norm(XTc, axis=1) + 1e-12
    )

    l2_move = np.linalg.norm(XT - X0, axis=1)
    l2_start = np.linalg.norm(X0, axis=1)

    print(f"mean corr(x0,xT): {corr.mean():.3f}")
    print(f"median ||xT-x0||_2: {np.median(l2_move):.3f}")
    print(f"median ||xT-x0||_2 / ||x0||_2: {np.median(l2_move / (l2_start + 1e-12)):.3f}")

    return {
        "name": name,
        "x0": x0_succ.detach().cpu(),
        "xT": xT_succ.detach().cpu(),
        "reached_frac": reached.float().mean().item(),
        "dims": res,
        "corr_mean": corr.mean(),
        "l2_move_median": np.median(l2_move),
    }


# ----------------------------
# Baselines before optimization
# ----------------------------

print("\nNatural-image baselines")
print("-" * 80)

baseline_results = []
baseline_results.append(estimate_dims("natural cats", X_cat[:N_SAMPLES]))
baseline_results.append(estimate_dims("natural noncats", X_noncat[:N_SAMPLES]))

# Single-cat tiny noise control from previous experiment.
x_one = X_cat[:1].repeat(N_SAMPLES, 1, 1, 1)
eps = 4 / 255
x_one_noise = (x_one + torch.empty_like(x_one).uniform_(-eps, eps)).clamp(0, 1)
baseline_results.append(estimate_dims("single cat + ±4/255 noise", x_one_noise))


# ----------------------------
# Run sanity-check initializations
# ----------------------------

conditions = [
    ("fixed gray + tiny jitter -> cat", lambda n: init_fixed_gray_jitter(n, gray=0.5, jitter=1/255)),
    ("random RGB constant -> cat", init_random_rgb_constant),
    ("smooth 4x4 noise -> cat", lambda n: init_smooth_noise(n, grid=4)),
    ("smooth 8x8 noise -> cat", lambda n: init_smooth_noise(n, grid=8)),
    ("iid pixel noise -> cat", init_iid_noise),
    ("natural cats -> cat", init_natural_cats),
    ("natural noncats -> cat", init_natural_noncats),
]

runs = []
for name, fn in conditions:
    runs.append(run_condition(name, fn, n=N_SAMPLES))


# ----------------------------
# Plot summary
# ----------------------------

names = [r["name"] for r in runs]
pr_final = [r["dims"]["xT"]["pr"] for r in runs]
twonn_final = [r["dims"]["xT"]["twonn"] for r in runs]
pr_init = [r["dims"]["x0"]["pr"] for r in runs]
twonn_init = [r["dims"]["x0"]["twonn"] for r in runs]

plt.figure(figsize=(12, 5))
x = np.arange(len(runs))
plt.bar(x - 0.2, pr_init, width=0.4, label="PR init")
plt.bar(x + 0.2, pr_final, width=0.4, label="PR final")
plt.xticks(x, names, rotation=60, ha="right")
plt.ylabel("Participation Ratio")
plt.title(f"PR before/after optimizing to {TARGET_NAME}")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.bar(x - 0.2, twonn_init, width=0.4, label="2NN init")
plt.bar(x + 0.2, twonn_final, width=0.4, label="2NN final")
plt.xticks(x, names, rotation=60, ha="right")
plt.ylabel("2NN estimated dimension")
plt.title(f"2NN before/after optimizing to {TARGET_NAME}")
plt.legend()
plt.tight_layout()
plt.show()


# ----------------------------
# Show image grids
# ----------------------------

def show_grid(title, imgs, nrow=8, max_imgs=32):
    imgs = imgs[:max_imgs].detach().cpu()
    grid = torchvision.utils.make_grid(imgs, nrow=nrow, padding=2)
    plt.figure(figsize=(12, 6))
    plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
    plt.axis("off")
    plt.title(title)
    plt.show()


def show_delta_grid(title, x0, xT, nrow=8, max_imgs=32):
    """
    Shows amplified absolute change so unchanged/weakly changed images are easy to spot.
    """
    x0 = x0[:max_imgs].detach().cpu()
    xT = xT[:max_imgs].detach().cpu()
    delta = (xT - x0).abs()

    # Normalize per displayed grid for visibility.
    m = delta.max().item()
    if m > 0:
        delta = delta / m

    grid = torchvision.utils.make_grid(delta, nrow=nrow, padding=2)
    plt.figure(figsize=(12, 6))
    plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
    plt.axis("off")
    plt.title(title + " | abs change, normalized")
    plt.show()


for r in runs:
    show_grid(r["name"] + " | initial x0", r["x0"], max_imgs=32)
    show_grid(r["name"] + " | final xT", r["xT"], max_imgs=32)
    show_delta_grid(r["name"], r["x0"], r["xT"], max_imgs=32)


# ----------------------------
# Interpretive printout
# ----------------------------

print("\nInterpretation guide:")
print("""
Expected if the noise-start PM estimate is initialization-driven:

  iid pixel noise -> cat:
    high PR, high 2NN

  random RGB constant / fixed gray -> cat:
    much lower PR and lower 2NN, unless optimizer itself creates diverse high-frequency junk

  smooth low-frequency starts -> cat:
    intermediate PR/2NN

  natural cats -> cat:
    near natural-cat PR/2NN or only modest inflation

  natural noncats -> cat:
    intermediate; it tests how much source-image residue remains while adding cat evidence

Important diagnostics:
  - PR(final) close to PR(init) suggests inherited initialization entropy.
  - high corr(x0,xT) suggests the optimized sample retains start structure.
  - high PR(highpass xT) suggests high-frequency nuisance structure.
  - if all conditions reach p(cat) >= threshold but dimensions differ wildly,
    then "PM dimensionality" is not just a property of the class region;
    it is strongly dependent on initialization + optimizer + stopping rule.

Code-specific sanity checks:
  - If 2NN prints duplicate cleanup for natural noncats/cats, something is still
    producing exact duplicates.
  - The condition initialization now happens once globally, not independently
    inside each batch.
  - The delta grids should make it obvious whether supposedly unchanged images
    actually changed.
""")

Device: cuda
Loading RobustBench model: Standard


Downloading...
From (original): https://drive.google.com/uc?id=1t98aEuzeTL8P7Kpd5DIrCoCL21BNZUhC
From (redirected): https://drive.google.com/uc?id=1t98aEuzeTL8P7Kpd5DIrCoCL21BNZUhC&confirm=t&uuid=d0f2cc24-0613-425f-8443-05fce1fb79ab
To: /content/models/cifar10/Linf/Standard.pt
100%|██████████| 292M/292M [00:06<00:00, 42.1MB/s]
100%|██████████| 170M/170M [00:13<00:00, 12.5MB/s]


Ambient dimension D = 3072
Loaded 4096 natural cats and 4096 noncats.

Natural-image baselines
--------------------------------------------------------------------------------
natural cats                        | n= 4096 | 2NN=    24.68 | PR=     9.66
natural noncats                     | n= 4096 | 2NN=    30.04 | PR=     9.27
single cat + ±4/255 noise           | n= 4096 | 2NN=   361.98 | PR=  1754.86

fixed gray + tiny jitter -> cat
Time: 60.1s
Reached threshold: 1.000
Mean p(cat): 0.999; median: 0.999
Unchanged-ish samples, ||xT-x0||_2 < 1e-6: 0.000
Median movement over all samples: 16.229435
fixed gray + tiny jitter -> cat | init x0 | n= 4096 | 2NN=   362.55 | PR=  1755.05
fixed gray + tiny jitter -> cat | final xT | n= 4096 | 2NN=   146.34 | PR=   465.07
fixed gray + tiny jitter -> cat | residual xT-x0 | n= 4096 | 2NN=   146.42 | PR=   465.18
fixed gray + tiny jitter -> cat | lowpass xT | n= 4096 | 2NN=    97.00 | PR=   252.70
fixed gray + tiny jitter -> cat | highpass-ish xT | n